In [13]:
import sys
from pathlib import Path
import subprocess

src_path = str(Path.cwd().parent / "src")
if src_path not in sys.path:
    sys.path.append(src_path)
from ini import initialisation

In [14]:
# Welche Store-Item-Kombinationen?
ITEMS_TO_MODEL = {
    "daily_smooth": {"store": 25, "item": 115611},
    "daily_erratic": {"store": 44, "item": 103520},
    "weekly_smooth": {"store": 24, "item": 1503844},
    "weekly_erratic": {"store": 51, "item": 1239986},
}

TEST_WEEKS = 4

print("✅ Konfiguration geladen")
print(f"   Test Period: {TEST_WEEKS} Wochen")

✅ Konfiguration geladen
   Test Period: 4 Wochen


In [15]:
def detect_gaps(df, date_col="ds", freq="D"):
    """Findet zeitliche Lücken."""
    df = df.sort_values(date_col)

    expected_dates = pd.date_range(
        start=df[date_col].min(), end=df[date_col].max(), freq=freq
    )

    actual_dates = set(df[date_col])
    missing_dates = sorted(set(expected_dates) - actual_dates)

    gap_info = {
        "has_gaps": len(missing_dates) > 0,
        "n_missing": len(missing_dates),
        "missing_dates": missing_dates[:10],
        "pct_missing": len(missing_dates) / len(expected_dates) * 100,
    }

    if gap_info["has_gaps"]:
        print(
            f"  ⚠️  Gaps: {gap_info['n_missing']} dates ({gap_info['pct_missing']:.1f}%)"
        )
    else:
        print("  ✅ No gaps")

    return gap_info


def fill_gaps(df, date_col="ds", target_col="y", freq="D"):
    """Füllt zeitliche Lücken."""
    df = df.sort_values(date_col).copy()

    full_range = pd.date_range(
        start=df[date_col].min(), end=df[date_col].max(), freq=freq
    )

    df = df.set_index(date_col).reindex(full_range).reset_index()
    df.columns = [date_col if c == "index" else c for c in df.columns]

    df[target_col] = df[target_col].fillna(0)

    if "unique_id" in df.columns:
        df["unique_id"] = df["unique_id"].fillna(method="ffill").fillna(method="bfill")

    print("  ✅ Filled with 0's")
    return df


def load_and_prepare(df, store, item, freq="D"):
    """Lädt und bereitet eine Store-Item-Kombination vor."""

    # Determine date column
    if "date" in df.columns:
        date_col = "date"
    elif "week_start" in df.columns:
        date_col = "week_start"
    else:
        df = df.reset_index()
        if "date" in df.columns:
            date_col = "date"
        elif "week_start" in df.columns:
            date_col = "week_start"
        else:
            date_col = df.columns[0]

    # Filter
    ts = df[(df["store_nbr"] == store) & (df["item_nbr"] == item)].copy()

    # Format
    ts[date_col] = pd.to_datetime(ts[date_col])
    ts = ts.sort_values(date_col)

    target_col = "unit_sales"

    ts = ts[[date_col, target_col]].rename(columns={date_col: "ds", target_col: "y"})

    ts["unique_id"] = f"store_{store}_item_{item}"

    print(f"📊 Loaded: {len(ts)} obs | {ts['ds'].min()} to {ts['ds'].max()}")
    print(f"   Mean: {ts['y'].mean():.2f}, Std: {ts['y'].std():.2f}")

    # Gap handling
    gap_info = detect_gaps(ts, date_col="ds", freq=freq)

    if gap_info["has_gaps"]:
        ts = fill_gaps(ts, date_col="ds", target_col="y", freq=freq)
        print(f"   After fill: {len(ts)} obs")

    return ts


def train_test_split(df, test_weeks=4):
    """Zeitbasierter Split."""
    cutoff = df["ds"].max() - pd.Timedelta(weeks=test_weeks)

    train = df[df["ds"] <= cutoff].copy()
    test = df[df["ds"] > cutoff].copy()

    print(f"✂️  Train: {len(train)} obs | Test: {len(test)} obs")

    return train, test


print("✅ Helper functions geladen")

✅ Helper functions geladen


In [16]:
PROJECT_ROOT = Path("..").resolve()

# Setze das Arbeitsverzeichnis auf das Hauptprojektverzeichnis
os.chdir(f"{PROJECT_ROOT}")

dfs = build_dataframes()

daily_smooth = dfs["smooth_daily"]  # Smooth  aus daily  Matrix
daily_erratic = dfs["erratic_daily"]  # Erratic aus daily  Matrix
weekly_smooth = dfs["smooth_weekly"]  # Smooth  aus weekly Matrix
weekly_erratic = dfs["erratic_weekly"]  # Erratic aus weekly Matrix

print("📂 Lade Daten...")


print(f"✅ Daily Smooth:   {daily_smooth.shape}")
print(f"✅ Daily Erratic:  {daily_erratic.shape}")
print(f"✅ Weekly Smooth:  {weekly_smooth.shape}")
print(f"✅ Weekly Erratic: {weekly_erratic.shape}")

Loading fact table …
Loading forecastability matrices …
Building DataFrames …
  smooth_daily         36,560 store-item pairs     39,523,827 rows
  erratic_daily        35,933 store-item pairs     36,705,401 rows
  smooth_weekly        47,196 store-item pairs     30,042,814 rows
  erratic_weekly       19,734 store-item pairs     10,182,367 rows
📂 Lade Daten...
✅ Daily Smooth:   (39523827, 12)
✅ Daily Erratic:  (36705401, 12)
✅ Weekly Smooth:  (30042814, 12)
✅ Weekly Erratic: (10182367, 12)


In [24]:
# Assumption: you already have PROJECT_ROOT like in your notebook
MLRUNS_DIR = PROJECT_ROOT / "mlruns"

# Local file based tracking inside the repo
mlflow.set_tracking_uri(f"file://{MLRUNS_DIR.as_posix()}")

# One experiment for your baseline iteration history
mlflow.set_experiment("favorita_baseline_store_item")

<Experiment: artifact_location='file:///Users/cristallagus/Desktop/GitHub/Public/retail_demand_analysis/mlruns/210836426628127013', creation_time=1778103785837, experiment_id='210836426628127013', last_update_time=1778103785837, lifecycle_stage='active', name='favorita_baseline_store_item', tags={}, trace_location=None, workspace='default'>

In [17]:
daily_smooth.head()

,id,date,store_nbr,item_nbr,unit_sales,onpromotion,year,dow,year_iso,week,week_start,month
0,12,2013-01-01,25,115611,1.0,<NA>,2013,1,2013,1,2012-12-31,2013-01-01
1,64,2013-01-01,25,215352,46.0,<NA>,2013,1,2013,1,2012-12-31,2013-01-01
2,76,2013-01-01,25,252970,3.0,<NA>,2013,1,2013,1,2012-12-31,2013-01-01
3,86,2013-01-01,25,261700,3.0,<NA>,2013,1,2013,1,2012-12-31,2013-01-01
4,103,2013-01-01,25,305080,10.0,<NA>,2013,1,2013,1,2012-12-31,2013-01-01


In [18]:
daily_smooth_clean = daily_smooth[daily_smooth["date"] < "2016-08-22"].copy()

In [44]:
def run_baseline_plotly(df, pattern, store, item, freq="D", season_length=7):
    """
    Baseline-Pipeline mit Plotly Visualisierungen.
    """
    current_dir = Path.cwd()
    root_path = next((p for p in [current_dir] + list(current_dir.parents) 
                     if (p / "src").exists() or (p / "configs").exists()), current_dir.parent)
    
    # --- MLFLOW PFAD-AUTONOMIE ---
    # Nutzt den dynamisch erkannten root_path, um mlruns im Projekt-Root zu finden
    ml_folder = root_path / "mlruns"
    
    # Sicherstellen, dass der Ordner existiert (verhindert Permission-Fehler)
    ml_folder.mkdir(parents=True, exist_ok=True)
    
    # Absoluter Pfad für MLflow Tracking (behebt Permission Denied: '/Users/patrickhederer')
    mlflow.set_tracking_uri(f"file://{ml_folder.absolute().as_posix()}")
    mlflow.set_experiment("favorita_baseline_store_item")
    run_name = f"{pattern}_store{store}_item{item}_season{season_length}"

    with mlflow.start_run(run_name=run_name):
        print("\n" + "=" * 70)
        print(f"🎯 PATTERN: {pattern.upper()}")
        print(f"   Store: {store} | Item: {item}")
        print("=" * 70)

        # MLflow parameters
        mlflow.log_params(
            {
                "1_pattern": pattern,
                "2_freq": freq,
                "3_store": store,
                "4_item": item,
                "5_season_length": season_length,
            }
        )

        # date_col = "date" if "date" in df.columns else "week_start"

        # 1. Prepare
        ts = load_and_prepare(df, store, item, freq=freq)

        # 2. Split
        train, test = train_test_split(ts, test_weeks=TEST_WEEKS)

        # 3. Train SARIMA
        print(f"\n🤖 Training SARIMA (season={season_length})...")
        model_sarima = StatsForecast(
            models=[AutoARIMA(season_length=season_length)],
            freq=freq,
            n_jobs=1,
        )
        model_sarima.fit(train)

        # 4. Forecast SARIMA
        horizon = len(test["ds"].unique())
        print(f"   Forecasting {horizon} periods...")
        forecasts_sarima = model_sarima.predict(h=horizon)

        test_sarima = test.merge(
            forecasts_sarima.reset_index(), on=["unique_id", "ds"], how="left"
        )

        actuals = test_sarima["y"].values
        preds_sarima = test_sarima["AutoARIMA"].values
        mask = ~np.isnan(preds_sarima) & ~np.isnan(actuals)

        mae_sarima = mean_absolute_error(actuals[mask], preds_sarima[mask])
        r2_sarima = r2_score(actuals[mask], preds_sarima[mask])
        print(f"   ✅ SARIMA MAE: {mae_sarima:.2f}")
        print(f"   ✅ SARIMA R²:  {r2_sarima:.2f}")

        # 5. Train Naive
        print(f"\n🤖 Training Seasonal Naive (season={season_length})...")
        model_naive = StatsForecast(
            models=[SeasonalNaive(season_length=season_length)],
            freq=freq,
            n_jobs=1,
        )
        model_naive.fit(train)

        forecasts_naive = model_naive.predict(h=horizon)
        test_naive = test.merge(
            forecasts_naive.reset_index(), on=["unique_id", "ds"], how="left"
        )

        preds_naive = test_naive["SeasonalNaive"].values
        mae_naive = mean_absolute_error(actuals[mask], preds_naive[mask])
        r2_naive = r2_score(actuals[mask], preds_naive[mask])
        print(f"   ✅ Naive MAE: {mae_naive:.2f}")
        print(f"   ✅ Naive R²:  {r2_naive:.2f}")

        # 6. Vergleich
        if mae_naive > 0:
            improvement = (mae_naive - mae_sarima) / mae_naive * 100
        else:
            # Falls Naive perfekt ist (MAE=0), setzen wir Improvement auf 0 
            # oder -100, falls SARIMA Fehler hat.
            improvement = 0.0 if mae_sarima == 0 else -100.0

        print("\n📊 RESULTS:")
        print(f"   SARIMA MAE:     {mae_sarima:.2f}")
        print(f"   Naive MAE:      {mae_naive:.2f}")
        print(f"   Improvement:    {improvement:+.1f}%")
        print(f"   SARIMA R²:      {r2_sarima:.2f}")
        print(f"   Naive R²:       {r2_naive:.2f}")

        # MLflow metrics
        mlflow.log_metrics(
            {
                "1_train_size": len(train),
                "2_test_size": len(test),
                "3_mae_naive": mae_naive,
                "4_r2_naive": r2_naive,
                "5_mae_primary": mae_sarima,
                "6_r2_primary": r2_sarima,
                "7_improvement_pct": improvement,
            }
        )

        # 7. PLOTLY VISUALIZATIONS

        COL_TRAIN = colors.blue_brand
        COL_ACTUAL = colors.blue_brand
        COL_SARIMA = colors.red_brand
        COL_NAIVE = colors.gold_brand
        COL_ZERO_LINE = colors.blue_brand

        # -------------------------------------------------------------------------
        # PLOT 1: Overview + Test Period Zoom
        # -------------------------------------------------------------------------

        fig = make_subplots(
            rows=2,
            cols=1,
            subplot_titles=(
                f"{pattern.upper()} - Forecast Comparison (Full Timeline)",
                f"Test Period Zoom (Improvement: {improvement:+.1f}%)",
            ),
            vertical_spacing=0.12,
            row_heights=[0.5, 0.5],
        )

        # Row 1: Full timeline
        fig.add_trace(
            go.Scatter(
                x=train["ds"],
                y=train["y"],
                mode="lines",
                name="Train",
                line={"color": COL_TRAIN, "width": 1},
                opacity=0.6,
                hovertemplate="Train<br>Date: %{x}<br>Sales: %{y:.2f}<extra></extra>",
            ),
            row=1,
            col=1,
        )

        fig.add_trace(
            go.Scatter(
                x=test_sarima["ds"],
                y=test_sarima["y"],
                mode="lines+markers",
                name="Test (Actual)",
                line={"color": COL_ACTUAL, "width": 2},
                marker={"size": 6},
                hovertemplate="Actual<br>Date: %{x}<br>Sales: %{y:.2f}<extra></extra>",
            ),
            row=1,
            col=1,
        )

        fig.add_trace(
            go.Scatter(
                x=test_sarima["ds"],
                y=test_sarima["AutoARIMA"],
                mode="lines+markers",
                name=f"SARIMA (MAE={mae_sarima:.2f})",
                line={"color": COL_NAIVE, "width": 1.8, "dash": "dot"},
                marker={"size": 4, "symbol": "diamond", "color": COL_NAIVE},
                hovertemplate="SARIMA<br>Date: %{x}<br>Forecast: %{y:.2f}<extra></extra>",
            ),
            row=1,
            col=1,
        )

        fig.add_trace(
            go.Scatter(
                x=test_naive["ds"],
                y=test_naive["SeasonalNaive"],
                mode="lines+markers",
                name=f"Naive (MAE={mae_naive:.2f})",
                line={"color": COL_NAIVE, "width": 1.6, "dash": "dot"},
                marker={"size": 5, "symbol": "diamond", "color": COL_NAIVE},
                hovertemplate="Naive<br>Date: %{x}<br>Forecast: %{y:.2f}<extra></extra>",
            ),
            row=1,
            col=1,
        )

        # Row 2: Zoom on test period
        fig.add_trace(
            go.Scatter(
                x=test_sarima["ds"],
                y=test_sarima["y"],
                mode="lines+markers",
                name="Actual",
                line={"color": COL_ACTUAL, "width": 2.5},
                marker={"size": 7, "symbol": "circle", "color": COL_ACTUAL},
                showlegend=False,
                hovertemplate="Actual<br>Date: %{x}<br>Sales: %{y:.2f}<extra></extra>",
            ),
            row=2,
            col=1,
        )

        fig.add_trace(
            go.Scatter(
                x=test_sarima["ds"],
                y=test_sarima["AutoARIMA"],
                mode="lines+markers",
                name="SARIMA",
                line={"color": COL_SARIMA, "width": 2, "dash": "dash"},
                marker={"size": 6, "symbol": "square", "color": COL_SARIMA},
                showlegend=False,
                hovertemplate="SARIMA<br>Date: %{x}<br>Forecast: %{y:.2f}<extra></extra>",
            ),
            row=2,
            col=1,
        )

        fig.add_trace(
            go.Scatter(
                x=test_naive["ds"],
                y=test_naive["SeasonalNaive"],
                mode="lines+markers",
                name="Naive",
                line={"color": COL_NAIVE, "width": 1.6, "dash": "dot"},
                marker={"size": 5, "symbol": "diamond", "color": COL_NAIVE},
                showlegend=False,
                hovertemplate="Naive<br>Date: %{x}<br>Forecast: %{y:.2f}<extra></extra>",
            ),
            row=2,
            col=1,
        )

        # Layout
        fig.update_xaxes(title_text="Date", row=1, col=1)
        fig.update_xaxes(title_text="Date", row=2, col=1)
        fig.update_yaxes(title_text="Sales", row=1, col=1)
        fig.update_yaxes(title_text="Sales", row=2, col=1)

        fig.update_layout(
            height=800,
            hovermode="x unified",
            showlegend=True,
            legend={
                "orientation": "h",
                "yanchor": "bottom",
                "y": 1.02,
                "xanchor": "right",
                "x": 1,
            },
        )

        fig.show()
        # Plot als MLflow Artifact speichern
        mlflow.log_figure(fig, "plots/forecast.html")

        # -------------------------------------------------------------------------
        # PLOT 2: Residual Analysis
        # -------------------------------------------------------------------------

        residuals_sarima = test_sarima["y"] - test_sarima["AutoARIMA"]
        residuals_naive = test_naive["y"] - test_naive["SeasonalNaive"]

        fig2 = make_subplots(
            rows=1,
            cols=2,
            subplot_titles=("Residuals Over Time", "Residual Distribution"),
            specs=[[{"type": "scatter"}, {"type": "histogram"}]],
        )

        # Residuals over time
        fig2.add_trace(
            go.Scatter(
                x=test_sarima["ds"],
                y=residuals_sarima,
                mode="markers",
                name="SARIMA",
                marker={"size": 8, "color": COL_SARIMA, "opacity": 0.75},
                hovertemplate="SARIMA Residual<br>Date: %{x}<br>Error: %{y:.2f}<extra></extra>",
            ),
            row=1,
            col=1,
        )

        fig2.add_trace(
            go.Scatter(
                x=test_naive["ds"],
                y=residuals_naive,
                mode="markers",
                name="Naive",
                marker={"size": 7, "color": COL_NAIVE, "opacity": 0.55},
                hovertemplate="Naive Residual<br>Date: %{x}<br>Error: %{y:.2f}<extra></extra>",
            ),
            row=1,
            col=1,
        )

        # Zero line
        fig2.add_hline(
            y=0,
            line_dash="dash",
            line_color=COL_ZERO_LINE,
            opacity=0.8,
            row=1,
            col=1,
        )

        # Histogram
        fig2.add_trace(
            go.Histogram(
                x=residuals_sarima.dropna(),
                name="SARIMA",
                marker_color=COL_SARIMA,
                opacity=0.7,
                nbinsx=15,
                hovertemplate="Bin: %{x}<br>Count: %{y}<extra></extra>",
            ),
            row=1,
            col=2,
        )

        fig2.add_trace(
            go.Histogram(
                x=residuals_naive.dropna(),
                name="Naive",
                marker_color=COL_NAIVE,
                opacity=0.5,
                nbinsx=15,
                hovertemplate="Bin: %{x}<br>Count: %{y}<extra></extra>",
            ),
            row=1,
            col=2,
        )

        # Zero line in histogram
        fig2.add_vline(
            x=0,
            line_dash="dash",
            line_color=COL_ZERO_LINE,
            opacity=0.8,
            row=1,
            col=2,
        )

        # Layout
        fig2.update_xaxes(title_text="Date", row=1, col=1)
        fig2.update_xaxes(title_text="Residual", row=1, col=2)
        fig2.update_yaxes(title_text="Residual (Actual - Predicted)", row=1, col=1)
        fig2.update_yaxes(title_text="Frequency", row=1, col=2)

        fig2.update_layout(
            height=400,
            barmode="overlay",
            showlegend=True,
            title_text=f"{pattern.upper()} - Residual Analysis",
        )

        fig2.show()
        # Plot als MLflow Artifact speichern
        mlflow.log_figure(fig2, "plots/residuals.html")

        pred_table = test_sarima[["ds", "y", "AutoARIMA"]].copy()
        pred_table["naive"] = test_naive["SeasonalNaive"]

        mlflow.log_table(pred_table, "predictions.json")

        mlflow.log_metrics({"train_size": len(train), "test_size": len(test)})

        mlflow.end_run()

        return {
            "pattern": pattern,
            "store": store,
            "item": item,
            "n_train": len(train),
            "n_test": len(test),
            "mae_sarima": mae_sarima,
            "mae_naive": mae_naive,
            "improvement_pct": improvement,
        }

    print("✅ Baseline function (Plotly) geladen")

In [20]:
daily_smooth_clean.head()

,id,date,store_nbr,item_nbr,unit_sales,onpromotion,year,dow,year_iso,week,week_start,month
0,12,2013-01-01,25,115611,1.0,<NA>,2013,1,2013,1,2012-12-31,2013-01-01
1,64,2013-01-01,25,215352,46.0,<NA>,2013,1,2013,1,2012-12-31,2013-01-01
2,76,2013-01-01,25,252970,3.0,<NA>,2013,1,2013,1,2012-12-31,2013-01-01
3,86,2013-01-01,25,261700,3.0,<NA>,2013,1,2013,1,2012-12-31,2013-01-01
4,103,2013-01-01,25,305080,10.0,<NA>,2013,1,2013,1,2012-12-31,2013-01-01


In [40]:
# RUN DAILY SMOOTH
# =============================================================================

result_daily_smooth = run_baseline_plotly(
    df=daily_smooth_clean,
    pattern="daily_smooth",
    store=ITEMS_TO_MODEL["daily_smooth"]["store"],
    item=ITEMS_TO_MODEL["daily_smooth"]["item"],
    freq="D",
    season_length=7,
)

2026/05/06 23:59:39 INFO mlflow.tracking.fluent: Experiment with name 'favorita_baseline_store_item' does not exist. Creating a new experiment.



🎯 PATTERN: DAILY_SMOOTH
   Store: 25 | Item: 115611
📊 Loaded: 1284 obs | 2013-01-01 00:00:00 to 2016-08-21 00:00:00
   Mean: 10.02, Std: 6.27
  ⚠️  Gaps: 45 dates (3.4%)
  ✅ Filled with 0's
   After fill: 1329 obs
✂️  Train: 1301 obs | Test: 28 obs

🤖 Training SARIMA (season=7)...


/var/folders/k8/7_ysp8813gz5xdcks_1dkqw40000gn/T/ipykernel_6158/926410061.py:43: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["unique_id"] = df["unique_id"].fillna(method="ffill").fillna(method="bfill")


   Forecasting 28 periods...
   ✅ SARIMA MAE: 3.94
   ✅ SARIMA R²:  -0.03

🤖 Training Seasonal Naive (season=7)...
   ✅ Naive MAE: 4.61
   ✅ Naive R²:  -0.60

📊 RESULTS:
   SARIMA MAE:     3.94
   Naive MAE:      4.61
   Improvement:    +14.6%
   SARIMA R²:      -0.03
   Naive R²:       -0.60


In [41]:
# =============================================================================
# CELL 7: RUN DAILY ERRATIC
# =============================================================================

result_daily_erratic = run_baseline_plotly(
    df=daily_erratic,
    pattern="daily_erratic",
    store=ITEMS_TO_MODEL["daily_erratic"]["store"],
    item=ITEMS_TO_MODEL["daily_erratic"]["item"],
    freq="D",
    season_length=7,
)


🎯 PATTERN: DAILY_ERRATIC
   Store: 44 | Item: 103520
📊 Loaded: 1580 obs | 2013-01-02 00:00:00 to 2017-08-15 00:00:00
   Mean: 8.73, Std: 7.26
  ⚠️  Gaps: 107 dates (6.3%)
  ✅ Filled with 0's
   After fill: 1687 obs
✂️  Train: 1659 obs | Test: 28 obs

🤖 Training SARIMA (season=7)...


/var/folders/k8/7_ysp8813gz5xdcks_1dkqw40000gn/T/ipykernel_6158/926410061.py:43: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["unique_id"] = df["unique_id"].fillna(method="ffill").fillna(method="bfill")


   Forecasting 28 periods...
   ✅ SARIMA MAE: 2.58
   ✅ SARIMA R²:  -0.15

🤖 Training Seasonal Naive (season=7)...
   ✅ Naive MAE: 2.82
   ✅ Naive R²:  -0.40

📊 RESULTS:
   SARIMA MAE:     2.58
   Naive MAE:      2.82
   Improvement:    +8.6%
   SARIMA R²:      -0.15
   Naive R²:       -0.40


In [42]:
# =============================================================================
# CELL 8: RUN WEEKLY SMOOTH
# =============================================================================

result_weekly_smooth = run_baseline_plotly(
    df=weekly_smooth,
    pattern="weekly_smooth",
    store=ITEMS_TO_MODEL["weekly_smooth"]["store"],
    item=ITEMS_TO_MODEL["weekly_smooth"]["item"],
    freq="W",
    season_length=52,
)


🎯 PATTERN: WEEKLY_SMOOTH
   Store: 24 | Item: 1503844
📊 Loaded: 993 obs | 2014-01-02 00:00:00 to 2017-08-15 00:00:00
   Mean: 248.84, Std: 69.46
  ⚠️  Gaps: 49 dates (25.9%)
  ✅ Filled with 0's
   After fill: 189 obs
✂️  Train: 185 obs | Test: 4 obs

🤖 Training SARIMA (season=52)...


/var/folders/k8/7_ysp8813gz5xdcks_1dkqw40000gn/T/ipykernel_6158/926410061.py:43: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["unique_id"] = df["unique_id"].fillna(method="ffill").fillna(method="bfill")


   Forecasting 4 periods...
   ✅ SARIMA MAE: 82.52
   ✅ SARIMA R²:  -8.94

🤖 Training Seasonal Naive (season=52)...
   ✅ Naive MAE: 33.58
   ✅ Naive R²:  -0.79

📊 RESULTS:
   SARIMA MAE:     82.52
   Naive MAE:      33.58
   Improvement:    -145.7%
   SARIMA R²:      -8.94
   Naive R²:       -0.79


In [45]:
# =============================================================================
# CELL 9: RUN WEEKLY ERRATIC (optional)
# =============================================================================

result_weekly_erratic = run_baseline_plotly(
    df=weekly_erratic,
    pattern="weekly_erratic",
    store=ITEMS_TO_MODEL["weekly_erratic"]["store"],
    item=ITEMS_TO_MODEL["weekly_erratic"]["item"],
    freq="W-MON",  # falls week_start = Montag
    season_length=52,
)


🎯 PATTERN: WEEKLY_ERRATIC
   Store: 51 | Item: 1239986
📊 Loaded: 184 obs | 2013-11-06 00:00:00 to 2017-08-04 00:00:00
   Mean: 1189.83, Std: 1413.94
  ⚠️  Gaps: 192 dates (98.5%)
  ✅ Filled with 0's
   After fill: 195 obs
✂️  Train: 191 obs | Test: 4 obs

🤖 Training SARIMA (season=52)...


/var/folders/k8/7_ysp8813gz5xdcks_1dkqw40000gn/T/ipykernel_6158/926410061.py:43: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["unique_id"] = df["unique_id"].fillna(method="ffill").fillna(method="bfill")


   Forecasting 4 periods...
   ✅ SARIMA MAE: 0.23
   ✅ SARIMA R²:  0.00

🤖 Training Seasonal Naive (season=52)...
   ✅ Naive MAE: 0.00
   ✅ Naive R²:  1.00

📊 RESULTS:
   SARIMA MAE:     0.23
   Naive MAE:      0.00
   Improvement:    -100.0%
   SARIMA R²:      0.00
   Naive R²:       1.00


In [47]:
# =============================================================================
# CELL 10: SUMMARY DASHBOARD
# =============================================================================

# Sammle Ergebnisse
results = [
    result_daily_smooth,
    result_daily_erratic,
    result_weekly_smooth,
    # result_weekly_erratic,
]

summary_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("📊 SUMMARY - ALL PATTERNS")
print("=" * 70)
print(summary_df.to_string(index=False))

# Interactive Bar Chart
fig = go.Figure()

x = summary_df["pattern"]

fig.add_trace(
    go.Bar(
        x=x,
        y=summary_df["mae_sarima"],
        name="SARIMA",
        marker_color="red",
        opacity=0.8,
        hovertemplate="%{x}<br>SARIMA MAE: %{y:.2f}<extra></extra>",
        text=summary_df["mae_sarima"].round(2),
        textposition="outside",
    )
)

fig.add_trace(
    go.Bar(
        x=x,
        y=summary_df["mae_naive"],
        name="Naive",
        marker_color="orange",
        opacity=0.6,
        hovertemplate="%{x}<br>Naive MAE: %{y:.2f}<extra></extra>",
        text=summary_df["mae_naive"].round(2),
        textposition="outside",
    )
)

# Improvement annotations
for i, row in summary_df.iterrows():
    fig.add_annotation(
        x=i,
        y=max(row["mae_sarima"], row["mae_naive"]) + 2,
        text=f"{row['improvement_pct']:+.1f}%",
        showarrow=False,
        font={
            "size": 14,
            "color": "green" if row["improvement_pct"] > 0 else "red",
            "family": "Arial Black",
        },
    )

fig.update_layout(
    title="Baseline Model Comparison - MAE by Pattern",
    xaxis_title="Pattern",
    yaxis_title="Mean Absolute Error (MAE)",
    barmode="group",
    height=500,
    hovermode="x unified",
    showlegend=True,
)

fig.show()


# --- CHIRURGISCHE PFAD-SICHERUNG ---
from pathlib import Path
base_path = root_path if 'root_path' in globals() else Path.cwd()
export_dir = base_path / "data" / "baseline_results"
export_dir.mkdir(parents=True, exist_ok=True)
export_file = export_dir / "baseline_summary.csv"

# Speichern
summary_df.to_csv(export_file, index=False)

print(f"\n✅ Summary saved to: {export_file}")


📊 SUMMARY - ALL PATTERNS
      pattern  store    item  n_train  n_test  mae_sarima  mae_naive  improvement_pct
 daily_smooth     25  115611     1301      28    3.935836   4.607143        14.571000
daily_erratic     44  103520     1659      28    2.578132   2.821429         8.623174
weekly_smooth     24 1503844      185       4   82.516821  33.580250      -145.730217



✅ Summary saved to: /Users/cristallagus/Desktop/GitHub/Public/retail_demand_analysis/data/baseline_results/baseline_summary.csv
